# FINPLE KR Price Metrics + Raw Daily Colab

Step 114-2Y extends the existing KR yfinance collector. Provider calls run only in this user-operated Colab. Six-digit KR ticker identity is preserved through 20-asset smoke, resumable 100-asset chunks, and combine.


In [ ]:
from pathlib import Path
import subprocess, sys

COLLECTION_REF = "codex/step114-2y-one-click-full-universe-app-preview"
REPO_ROOT = Path('/content/FINPLE')
COLLECTOR_FILE = REPO_ROOT / 'scripts' / 'build_kr_price_metrics_overlay_chunked.py'
COLLECTOR_MODULE = 'scripts.build_kr_price_metrics_overlay_chunked'
COMBINE_MODULE = 'scripts.combine_kr_price_metrics_chunks'
REQUIRED_COLLECTOR_OPTIONS = {'--out-raw', '--retrieved-at', '--resume'}

def run_visible(command, label, cwd):
    resolved_cwd = Path(cwd).resolve()
    print({'label': label, 'command': [str(part) for part in command], 'cwd': str(resolved_cwd)}, flush=True)
    try:
        result = subprocess.run(
            command, cwd=resolved_cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False
        )
    except OSError as exc:
        print({'returnCode': None}, flush=True)
        print('--- stdout ---\n', flush=True)
        print(f'--- stderr ---\n{exc}', flush=True)
        raise RuntimeError(f'{label} could not start') from exc
    print({'returnCode': result.returncode}, flush=True)
    print('--- stdout ---', flush=True)
    print(result.stdout or '', end='' if (result.stdout or '').endswith('\n') else '\n', flush=True)
    print('--- stderr ---', flush=True)
    print(result.stderr or '', end='' if (result.stderr or '').endswith('\n') else '\n', flush=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit status {result.returncode}')
    return result

def checkout_and_preflight(collection_ref, collector_file):
    run_visible(['git', 'fetch', '--depth', '1', 'origin', collection_ref], 'fetch requested ref', REPO_ROOT)
    run_visible(['git', 'checkout', '--detach', 'FETCH_HEAD'], 'detached FETCH_HEAD checkout', REPO_ROOT)
    head = run_visible(['git', 'rev-parse', 'HEAD'], 'resolved HEAD', REPO_ROOT).stdout.strip()
    status = run_visible(['git', 'status', '--short', '--branch'], 'checkout status', REPO_ROOT).stdout.strip()
    collector_exists = collector_file.is_file()
    print({'requestedRef': collection_ref, 'head': head, 'status': status, 'collector': str(collector_file), 'collectorExists': collector_exists})
    if not collector_exists:
        raise FileNotFoundError(f'Required collector is missing at {head}: {collector_file}')
    run_visible([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance', 'pandas'], 'install collector dependencies', REPO_ROOT)
    help_result = run_visible([sys.executable, '-m', COLLECTOR_MODULE, '--help'], 'collector module CLI preflight', REPO_ROOT)
    missing_options = sorted(option for option in REQUIRED_COLLECTOR_OPTIONS if option not in help_result.stdout)
    if missing_options:
        raise RuntimeError(f'Collector at {head} is too old; missing CLI options: {missing_options}')
    run_visible([sys.executable, '-m', COMBINE_MODULE, '--help'], 'combine module CLI preflight', REPO_ROOT)
    print({'collectorCliPreflight': 'ok', 'requiredOptions': sorted(REQUIRED_COLLECTOR_OPTIONS), 'module': COLLECTOR_MODULE})

!rm -rf /content/FINPLE
!git clone --depth 1 https://github.com/vip930sw/FINPLE.git /content/FINPLE
%cd /content/FINPLE
checkout_and_preflight(COLLECTION_REF, COLLECTOR_FILE)


In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import csv, json, shutil

AS_OF = date.today().isoformat()
RUN_TAG = AS_OF.replace('-', '')
RETRIEVED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
UNIVERSE = Path('src/data/tickers/finple_app_candidates_6000_balanced_v1.csv')
ROOT = Path('/content/finple-step114-2y-kr') / RUN_TAG
SMOKE_DIR = ROOT / 'smoke'
CHUNK_DIR = ROOT / 'chunks'
COMBINED_DIR = ROOT / 'combined'
ROOT.mkdir(parents=True, exist_ok=True)
for path in [CHUNK_DIR, COMBINED_DIR]:
    path.mkdir(parents=True, exist_ok=True)
if SMOKE_DIR.exists():
    root_resolved = ROOT.resolve()
    smoke_resolved = SMOKE_DIR.resolve()
    if smoke_resolved.parent != root_resolved or smoke_resolved.name != 'smoke':
        raise RuntimeError(f'Unsafe smoke cleanup target: {smoke_resolved}')
    shutil.rmtree(smoke_resolved)
SMOKE_DIR.mkdir(parents=True, exist_ok=False)
print({'smokeOutputMode': 'fresh', 'smokeDirectory': str(SMOKE_DIR)})

with UNIVERSE.open('r', encoding='utf-8-sig', newline='') as handle:
    universe_rows = list(csv.DictReader(handle))
KR_TOTAL = sum(row['market'] == 'KR' for row in universe_rows)
print({'asOf': AS_OF, 'retrievedAt': RETRIEVED_AT, 'krCandidates': KR_TOTAL, 'root': str(ROOT)})


## 1. Required 20-asset smoke


In [ ]:
smoke_prefix = SMOKE_DIR / 'kr_0000_0020'
run_visible([
    sys.executable, '-m', COLLECTOR_MODULE,
    '--input', str(UNIVERSE),
    '--out-runtime', str(smoke_prefix) + '_metrics.csv',
    '--out-audit', str(smoke_prefix) + '_audit.csv',
    '--out-summary', str(smoke_prefix) + '_summary.json',
    '--out-raw', str(smoke_prefix) + '_raw_daily.csv',
    '--as-of', AS_OF, '--start', '0', '--limit', '20',
    '--checkpoint-every', '5', '--retrieved-at', RETRIEVED_AT,
], 'KR 20-asset smoke', REPO_ROOT)
with open(str(smoke_prefix) + '_summary.json', encoding='utf-8') as handle:
    smoke_summary = json.load(handle)
assert smoke_summary['processed_count'] == 20
assert smoke_summary['raw_daily_asset_count'] == 20, smoke_summary
with open(str(smoke_prefix) + '_raw_daily.csv', encoding='utf-8-sig', newline='') as handle:
    smoke_raw = list(csv.DictReader(handle))
assert all(len(row['ticker']) == 6 and row['ticker'].isdigit() for row in smoke_raw)
print(json.dumps(smoke_summary, indent=2))


## 2. Full KR collection in resumable 100-asset chunks


In [ ]:
for start in range(0, KR_TOTAL, 100):
    end = min(start + 100, KR_TOTAL)
    prefix = CHUNK_DIR / f'kr_{start:04d}_{end:04d}'
    run_visible([
        sys.executable, '-m', COLLECTOR_MODULE,
        '--input', str(UNIVERSE),
        '--out-runtime', str(prefix) + '_metrics.csv',
        '--out-audit', str(prefix) + '_audit.csv',
        '--out-summary', str(prefix) + '_summary.json',
        '--out-raw', str(prefix) + '_raw_daily.csv',
        '--as-of', AS_OF, '--start', str(start), '--limit', str(end - start),
        '--checkpoint-every', '25', '--retrieved-at', RETRIEVED_AT, '--resume',
    ], f'KR chunk {start}:{end}', REPO_ROOT)
print('KR chunks complete:', len(list(CHUNK_DIR.glob('*_metrics.csv'))))


## 3. Combine and reconcile


In [ ]:
KR_METRICS = COMBINED_DIR / 'kr_price_metrics_overlay.csv'
KR_RAW = COMBINED_DIR / 'kr_raw_daily_prices.csv'
KR_SUMMARY = COMBINED_DIR / 'kr_price_metrics_summary.json'
run_visible([
    sys.executable, '-m', COMBINE_MODULE,
    '--pattern', str(CHUNK_DIR / '*_metrics.csv'),
    '--out-runtime', str(KR_METRICS), '--out-summary', str(KR_SUMMARY),
    '--raw-pattern', str(CHUNK_DIR / '*_raw_daily.csv'), '--out-raw', str(KR_RAW),
], 'combine KR chunks', REPO_ROOT)
with KR_SUMMARY.open(encoding='utf-8') as handle:
    combined_summary = json.load(handle)
assert combined_summary['row_count'] == KR_TOTAL
assert combined_summary['rawDailyAssetCount'] <= KR_TOTAL
print(json.dumps(combined_summary, indent=2))
print('One-Click inputs:', KR_RAW, KR_METRICS)


In [ ]:
DOWNLOAD_OUTPUTS = False
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    for target in [KR_RAW, KR_METRICS, KR_SUMMARY]:
        files.download(str(target))
else:
    print('Set DOWNLOAD_OUTPUTS=True after checking reconciliation.')
